# N2 — Contractual Forecast

## Decision question

What does the cash position look like if every outstanding invoice follows its
contractual due date—and where is that assumption fragile?


In [ ]:
from pathlib import Path
import json
import sys

# Find the public package locally. A fresh Colab runtime downloads the same
# participant-safe assets from the repository.
for candidate in [Path.cwd(), *Path.cwd().parents]:
    for source_candidate in (candidate / 'src', candidate / 'CFOPackV002' / 'src'):
        if (source_candidate / 'workshop_bootstrap.py').exists():
            sys.path.insert(0, str(source_candidate))
            break

try:
    from workshop_bootstrap import bootstrap
except ImportError:
    from urllib.request import urlopen
    bootstrap_url = (
        'https://raw.githubusercontent.com/VinayaSharada/'
        'KateelLearningDemosToStudents/cfopack-v002-v2.1.0-beta.1/CFOPackV002/src/workshop_bootstrap.py'
    )
    namespace = {}
    exec(compile(urlopen(bootstrap_url).read(), bootstrap_url, 'exec'), namespace)
    bootstrap = namespace['bootstrap']

ROOT, OUTPUT_DIR = bootstrap()
from cfopack_v002 import (
    analyze_fx,
    default_decisions,
    load_inputs,
    load_manifest,
    reveal_team_shock,
    run_pipeline,
)
import workshop_visuals as viz
import pandas as pd
try:
    from IPython.display import Markdown, display
except ImportError:
    # Keep the notebooks runnable from a minimal local Python environment as
    # well as Colab/Jupyter. Rich notebook rendering remains the default.
    def Markdown(value):
        return value

    def display(value):
        print(value)

manifest = load_manifest(ROOT / 'config' / 'scenario_manifest.json')
decision_file = OUTPUT_DIR / 'N0_team_decisions.json'
if decision_file.exists():
    DECISIONS = json.loads(decision_file.read_text(encoding='utf-8'))
else:
    DECISIONS = default_decisions(manifest)


In [ ]:
data = load_inputs(ROOT / 'data' / 'synthetic')
viz.data_snapshot(data, OUTPUT_DIR, 'N2')


In [ ]:
summary = run_pipeline(ROOT, OUTPUT_DIR, DECISIONS)
print(f"Scenario {summary['scenario_version']} calculated for {DECISIONS['team_name']} (model cache: {'hit' if summary['model_cache_hit'] else 'rebuilt'})")


## Inspect the contractual cash path


In [ ]:
forecast = pd.read_csv(OUTPUT_DIR / 'N2_contractual_forecast.csv', parse_dates=['date'])
minimum = manifest['minimum_liquidity']
display(forecast[['day', 'date', 'receipts', 'total_outflows', 'closing_cash', 'below_minimum']])
viz.forecast_chart(forecast, minimum, OUTPUT_DIR, 'N2_contractual_forecast.png', 'Contractual 30-day cash forecast')
low = forecast.loc[forecast['closing_cash'].idxmin()]
print(f"Contractual minimum: ${low['closing_cash']:,.0f} on Day {int(low['day'])}")


## Assumption challenge

List the contractual receipts that matter most to the closing balance. What
evidence would justify treating their due dates as cash dates?


### Before moving on

Record your interpretation in the participant workbook. Do not copy a chart
without also recording the assumption and decision it supports.
